In [ ]:
!pip install -q huggingface_hub
import os
from huggingface_hub import login
login(token=os.environ["HF_TOKEN"])

In [2]:
!pip -q install -U "transformers>=4.57.6" "trl>=0.10.1" "peft>=0.12.0" "accelerate>=0.33.0" "bitsandbytes>=0.43.1" "datasets>=2.20.0" "scikit-learn>=1.4.0" "huggingface_hub>=1.01"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 121.7 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 47.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 41.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.5/527.5 kB 47.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 137.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 52.0 MB/s eta 0:00:00:00:0100:01


In [3]:
from dataclasses import dataclass
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive/')

import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from sklearn.model_selection import train_test_split
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    BitsAndBytesConfig,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training

pd.set_option('display.max_colwidth', 180)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

Mounted at /content/drive/


In [4]:
@dataclass
class Config:
    # Model
    base_model = 'google/gemma-2-9b-it'

    # Flags
    run_training = True
    use_swap_aug = True
    tta = True

    # Paths
    adapter_path = './adapter'
    submission_path = './submission.csv'

    # Data
    max_seq_len = 2048
    val_size = 0.05
    random_state = 42

    # Training
    batch_size = 4
    num_epochs = 1
    per_device_batch_size = 1
    grad_accum = 8
    learning_rate = 2e-4
    warmup_ratio = 0.03
    logging_steps = 10
    save_steps = 200

    # LoRA
    lora_r = 16
    lora_alpha = 32
    lora_dropout = 0.05
    lora_target_modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'up_proj', 'down_proj', 'gate_proj']

cfg = Config()

In [5]:
def process_text(x: str) -> str:
    # eval handles Python-style null values that json.loads cannot parse
    return " ".join(" ".join(eval(x, {"null": ""})).split())

def to_label(row) -> int:
    # 0 = model_a wins, 1 = model_b wins, 2 = tie
    if row['winner_model_a'] == 1: return 0
    if row['winner_model_b'] == 1: return 1
    return 2


In [6]:
import pandas as pd
import math
import json

train_df = pd.read_csv('/content/drive/MyDrive/llm-finetuning/train.csv')
test_df  = pd.read_csv('/content/drive/MyDrive/llm-finetuning/test.csv')

def flatten_text(x):
    if x is None:
        return ""
    if isinstance(x, float) and math.isnan(x):
        return ""
    if isinstance(x, str):
        s = x.strip()
        if not s:
            return ""
        try:
            x = json.loads(s)
        except Exception:
            return s 
        
    if isinstance(x, dict):
        x = list(x.values())
    if isinstance(x, (list, tuple)):
        parts = []
        for v in x:
            if v is None:
                continue
            if isinstance(v, float) and math.isnan(v):
                continue
            parts.append(str(v).strip())
        return " ".join(p for p in parts if p)
    return str(x).strip()

def to_scalar_text(x):
    if isinstance(x, pd.Series):
        return x.iloc[0]
    if x is None:
        return ""
    if isinstance(x, float) and np.isnan(x):
        return ""
    return str(x)

def tokenize(tokenizer, prompt, response_a, response_b, max_length=cfg.max_seq_len):
    prompt = flatten_text(prompt)
    response_a = flatten_text(response_a)
    response_b = flatten_text(response_b)

    chunk = max_length // 3
    p_ids  = tokenizer('<prompt>: ' + prompt, max_length=chunk, truncation=True, padding=False).input_ids
    ra_ids = tokenizer('<response_a>: ' + response_a, max_length=chunk, truncation=True, padding=False).input_ids
    rb_ids = tokenizer('<response_b>: ' + response_b, max_length=chunk, truncation=True, padding=False).input_ids

    input_ids = p_ids + ra_ids + rb_ids
    return {'input_ids': input_ids, 'attention_mask': [1] * len(input_ids)}

for df in [train_df, test_df]:
    for col in ['prompt', 'response_a', 'response_b']:
        df[col] = df[col].apply(flatten_text)

train_df['label'] = train_df.apply(to_label, axis=1)
print('Label distribution:\n', train_df['label'].value_counts())

if cfg.use_swap_aug:
    # double training data by swapping A/B and flipping labels
    swapped = train_df.copy()
    swapped['response_a'], swapped['response_b'] = train_df['response_b'].copy(), train_df['response_a'].copy()
    swapped['label'] = swapped['label'].map({0: 1, 1: 0, 2: 2})
    train_df = pd.concat([train_df, swapped], ignore_index=True)
    print(f'After swap aug: {len(train_df)} rows')

Label distribution:
 label
0    20064
1    19652
2    17761
Name: count, dtype: int64
After swap aug: 114954 rows


In [7]:
for col in ['prompt', 'response_a', 'response_b']:
      print(col)
      print(train_df[col].map(type).value_counts().head(10))

prompt
prompt
<class 'str'>    114954
Name: count, dtype: int64
response_a
response_a
<class 'str'>    114954
Name: count, dtype: int64
response_b
response_b
<class 'str'>    114954
Name: count, dtype: int64


In [10]:
tokenizer = AutoTokenizer.from_pretrained(cfg.base_model)
tokenizer.padding_side = 'right'
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def make_dataset(df):
    records = []
    for _, row in df.iterrows():
        enc = tokenize(tokenizer, row['prompt'], row['response_a'], row['response_b'])
        enc['labels'] = int(row['label'])
        records.append(enc)
    return Dataset.from_list(records)

train_part, val_part = train_test_split(
    train_df.copy(),
    test_size=cfg.val_size,
    random_state=cfg.random_state,
    stratify=train_df['label'],
)

ds_train = make_dataset(train_part)
ds_val = make_dataset(val_part)
print(f'Train: {len(ds_train)} | Val: {len(ds_val)}')

Train: 109206 | Val: 5748


In [11]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model = AutoModelForSequenceClassification.from_pretrained(
    cfg.base_model,
    num_labels=3,
    quantization_config=bnb_config,
    device_map='auto',
)

model.config.use_cache = False
model.config.pad_token_id = tokenizer.pad_token_id

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=cfg.lora_r,
    lora_alpha=cfg.lora_alpha,
    lora_dropout=cfg.lora_dropout,
    bias='none',
    task_type='SEQ_CLS',
    target_modules=cfg.lora_target_modules,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/464 [00:00<?, ?it/s]

Gemma2ForSequenceClassification LOAD REPORT from: google/gemma-2-9b-it
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 54,028,800 || all params: 9,295,745,536 || trainable%: 0.5812


In [ ]:
training_args = TrainingArguments(
    output_dir='trainer_tmp',
    num_train_epochs=cfg.num_epochs,
    per_device_train_batch_size=cfg.per_device_batch_size,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=cfg.grad_accum,
    learning_rate=cfg.learning_rate,
    warmup_ratio=cfg.warmup_ratio,
    logging_steps=cfg.logging_steps,
    eval_strategy='steps',
    eval_steps=cfg.save_steps,
    save_steps=cfg.save_steps,
    save_total_limit=1,
    bf16=True,
    gradient_checkpointing=True,
    report_to='none',
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=ds_train,
    eval_dataset=ds_val,
    data_collator=DataCollatorWithPadding(tokenizer),
)
trainer.train()

Path(cfg.adapter_path).mkdir(parents=True, exist_ok=True)
trainer.model.save_pretrained(cfg.adapter_path)
tokenizer.save_pretrained(cfg.adapter_path)
print('Adapter saved to:', cfg.adapter_path)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Step,Training Loss,Validation Loss


In [ ]:
# load tokenizer and adapter from saved path (works standalone without running training cell)
tokenizer = AutoTokenizer.from_pretrained(cfg.adapter_path)
tokenizer.padding_side = 'right'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)
base_model = AutoModelForSequenceClassification.from_pretrained(
    cfg.base_model,
    num_labels=3,
    quantization_config=bnb_config,
    device_map='auto',
)
base_model.config.pad_token_id = tokenizer.pad_token_id
model = PeftModel.from_pretrained(base_model, cfg.adapter_path)
model.eval()

device = torch.device('cuda')

In [ ]:
def make_test_data(src_df, swap=False):
    ids, input_ids, masks = [], [], []
    for _, row in src_df.iterrows():
        ra = row['response_b'] if swap else row['response_a']
        rb = row['response_a'] if swap else row['response_b']
        enc = tokenize(tokenizer, row['prompt'], ra, rb)
        ids.append(row['id'])
        input_ids.append(enc['input_ids'])
        masks.append(enc['attention_mask'])
    df = pd.DataFrame({'id': ids, 'input_ids': input_ids, 'attention_mask': masks})
    df['length'] = df['input_ids'].apply(len)
    return df

@torch.no_grad()
@torch.cuda.amp.autocast()
def inference(df):
    # sort by length to minimise padding waste within each batch
    df = df.sort_values('length', ascending=False).reset_index(drop=True)
    a_win, b_win, tie = [], [], []
    for i in range(0, len(df), cfg.batch_size):
        batch  = df.iloc[i:i + cfg.batch_size]
        inputs = tokenizer.pad(
            {'input_ids': batch['input_ids'].tolist(), 'attention_mask': batch['attention_mask'].tolist()},
            padding=True,
            return_tensors='pt',
        ).to(device)
        proba = model(**inputs).logits.softmax(-1).cpu()
        a_win.extend(proba[:, 0].tolist())
        b_win.extend(proba[:, 1].tolist())
        tie.extend(proba[:, 2].tolist())
    df['winner_model_a'] = a_win
    df['winner_model_b'] = b_win
    df['winner_tie']     = tie
    return df

In [ ]:
data      = make_test_data(test_df)
result_df = inference(data)
proba     = result_df[['winner_model_a', 'winner_model_b', 'winner_tie']].values

if cfg.tta:
    aug_data      = make_test_data(test_df, swap=True)
    tta_result_df = inference(aug_data)
    # A/B are swapped in aug, so flip them back before averaging
    tta_proba = tta_result_df[['winner_model_b', 'winner_model_a', 'winner_tie']].values
    proba     = (proba + tta_proba) / 2

sub = pd.DataFrame({
    'id':             result_df['id'],
    'winner_model_a': proba[:, 0],
    'winner_model_b': proba[:, 1],
    'winner_tie':     proba[:, 2],
})
sub.to_csv(cfg.submission_path, index=False)
print('Saved:', cfg.submission_path)
sub.head()